<a href="https://www.kaggle.com/code/kedhareswernaidu/moonknight?scriptVersionId=229060647" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Like Paintings

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os

monet_jpg_dir = "/kaggle/input/c/gan-getting-started/monet_jpg"  
photo_jpg_dir = "/kaggle/input/c/gan-getting-started/photo_jpg" 

# Function to load and preprocess an image
def load_and_preprocess_image(filename, img_size=(256, 256)):
    """
    Loads an image file, decodes it, resizes it and scales pixel values.
    """
    image = tf.io.read_file(filename)  # Read the file
    image = tf.image.decode_jpeg(image, channels=3)  # Decode JPEG, ensuring 3 channels (RGB)
    image = tf.image.resize(image, img_size)  # Resize to desired size
    image = tf.cast(image, tf.float32) / 255.0  # Scale pixel values to [0,1]
    return image

# Function to display a batch of images from a dataset
def show_images(dataset, title, num_images=5):
    """
    Plots a few images from the given dataset.
    """
    plt.figure(figsize=(15, 3))
    for i, image in enumerate(dataset.take(num_images)):
        plt.subplot(1, num_images, i + 1)
        plt.imshow(image.numpy())
        plt.title(title)
        plt.axis('off')
    plt.show()

# Create datasets of file paths for Monet paintings and photographs
monet_jpg_files = tf.data.Dataset.list_files(os.path.join(monet_jpg_dir, "*.jpg"), shuffle=True)
photo_jpg_files = tf.data.Dataset.list_files(os.path.join(photo_jpg_dir, "*.jpg"), shuffle=True)

# Map file paths to actual image tensors
monet_dataset = monet_jpg_files.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
photo_dataset = photo_jpg_files.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

# Visualize a sample of images from each dataset
print("Visualizing sample Monet paintings:")
show_images(monet_dataset, "Monet Painting")

print("Visualizing sample Photographs:")
show_images(photo_dataset, "Photograph")


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# Define some global parameters
IMG_HEIGHT = 256
IMG_WIDTH = 256
CHANNELS = 3
NOISE_DIM = 100
BATCH_SIZE = 32

# Generator Model
def build_generator():
    model = tf.keras.Sequential(name="Generator")
    # Input: noise vector
    model.add(layers.Dense(16 * 16 * 256, use_bias=False, input_shape=(NOISE_DIM,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    
    model.add(layers.Reshape((16, 16, 256)))  # Reshape to a small feature map
    
    # Upsampling blocks
    model.add(layers.Conv2DTranspose(128, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    
    model.add(layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    
    model.add(layers.Conv2DTranspose(32, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    
    # Final layer to get desired output shape (256x256x3)
    model.add(layers.Conv2DTranspose(CHANNELS, kernel_size=5, strides=2, padding='same', use_bias=False, activation='tanh'))
    
    return model

# Discriminator Model
def build_discriminator():
    model = tf.keras.Sequential(name="Discriminator")
    # Input: image of shape (256,256,3)
    model.add(layers.Conv2D(32, kernel_size=5, strides=2, padding='same', input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))
    
    model.add(layers.Conv2D(64, kernel_size=5, strides=2, padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))
    
    model.add(layers.Conv2D(128, kernel_size=5, strides=2, padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))
    
    model.add(layers.Conv2D(256, kernel_size=5, strides=2, padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))
    
    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    return model

# Instantiate the models
generator = build_generator()
discriminator = build_discriminator()

# Print summaries for inspection
generator.summary()
discriminator.summary()

# Define loss and optimizers
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

In [ ]:
import time
import matplotlib.pyplot as plt

# Set training parameters
EPOCHS = 100
NUM_EXAMPLES_TO_GENERATE = 16
seed = tf.random.normal([NUM_EXAMPLES_TO_GENERATE, NOISE_DIM])

# Prepare the training dataset using Monet paintings (real data for our GAN)
monet_dataset_train = monet_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])
    
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)
    
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))
    
    return gen_loss, disc_loss

def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        # Scale images from [-1, 1] to [0, 1] if tanh is used
        plt.imshow((predictions[i] + 1.0) / 2.0)
        plt.axis('off')
    plt.suptitle(f'Epoch {epoch+1}')
    plt.show()

def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()
        
        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)
        
        print(f'Epoch {epoch+1}, Generator Loss: {gen_loss.numpy():.4f}, Discriminator Loss: {disc_loss.numpy():.4f}, Time: {time.time() - start:.2f} sec')
        
        # Visualize every 10 epochs only
        if (epoch + 1) % 10 == 0:
            generate_and_save_images(generator, epoch, seed)

# Start training
train(monet_dataset_train, EPOCHS)